# Type Hinting, Dataclasses & Typing Contracts: Beginner Guide

### 🌟 What Are Type Hints, Dataclasses & Typing Contracts?
Modern Python allows you to add optional **Type Hints** to function parameters and return values. Combined with **`@dataclass`**, you can build structured, type-safe data models with automatic string representations and comparison methods in minimal code.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Type Annotations**: Covers `Union`, `Optional`, and `Callable`.
- **Generics**: Covers generic containers (`List`, `Dict`) and `TypeVar`.
- **Structural Typing**: Covers `typing.Protocol` (static duck typing).
- **Data Modeling**: Covers `@dataclass` and immutable `@dataclass(frozen=True)`.
- **Advanced Constraints**: Covers `Literal`, `Final`, and `TypedDict`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol, Literal, Final, TypedDict, Callable, TypeVar

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from ../data/raw_transactions.csv


### 🔹 Type Annotation: `Union[A, B]`
Annotates parameters or return types that can accept multiple distinct candidate types. Checking and validating data types prevents subtle runtime errors and ensures subsequent mathematical or string operations behave properly. **Tip:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.

**Syntax:** `def parse(val: Union[int, float, str]) -> float: ...`


In [2]:
def parse_num(val: Union[int, float, str]) -> float:
    return float(val)

print('Union parameter parsed:', parse_num(transactions[0]['transaction_amount']))

Union parameter parsed: 1216.33


### 🔹 Type Annotation: `Optional[T]`
Shorthand for `Union[T, None]`, representing optional arguments that can be None. Checking and validating data types prevents subtle runtime errors and ensures subsequent mathematical or string operations behave properly. **Tip:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.

**Syntax:** `def get_tx(tx_id: str) -> Optional[dict]: ...`


In [3]:
def find_transaction(tx_id: str) -> Optional[dict]:
    return transactions[0] if tx_id == transactions[0]['transaction_id'] else None

print('Optional return:', find_transaction('TX110686')['card_type'] if find_transaction('TX110686') else 'None')

Optional return: Visa


### 🔹 Type Annotation: `Callable[[Args], Return]`
Annotates higher-order functions expecting callback functions as arguments. Checking and validating data types prevents subtle runtime errors and ensures subsequent mathematical or string operations behave properly. **Tip:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.

**Syntax:** `def apply(fn: Callable[[float], float], amt: float) -> float: ...`


In [4]:
def apply_fee(fee_calc: Callable[[float], float], amt: float) -> float:
    return fee_calc(amt)

print('Callable output:', apply_fee(lambda a: a * 0.02, 500.0))

Callable output: 10.0


### 🔹 Generic Containers: `List[T]` & `Dict[K, V]`
Annotates collection element types for static type analysis. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Use `dict.get(key, default)` when looking up keys that might not exist, avoiding unexpected `KeyError` crashes.

**Syntax:** `def get_ids(rows: List[Dict[str, str]]) -> List[str]: ...`


In [5]:
def extract_ids(records: List[Dict[str, str]]) -> List[str]:
    return [r['transaction_id'] for r in records]

print('Generic annotated extraction:', extract_ids(transactions[:3]))

Generic annotated extraction: ['TX110686', 'TX107170', 'TX108328']


### 🔹 Generic Type Variables: `TypeVar`
Defines type variables enabling generic reusable functions preserving input types. Checking and validating data types prevents subtle runtime errors and ensures subsequent mathematical or string operations behave properly. **Tip:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.

**Syntax:** `T = TypeVar('T')`


In [6]:
T = TypeVar('T')
def first_item(items: List[T]) -> T:
    return items[0]

print('Generic TypeVar output:', first_item(transactions[:3])['transaction_id'])

Generic TypeVar output: TX110686


### 🔹 Structural Typing with `Protocol` (Static Duck Typing)
Defines structural interfaces where classes conform implicitly without inheritance. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `class Payable(Protocol): def get_amount(self) -> float: ...`


In [7]:
class Payable(Protocol):
    def get_amount(self) -> float: ...

class Invoice:
    def __init__(self, amt): self.amt = amt
    def get_amount(self) -> float: return self.amt

def process_payment(item: Payable) -> str:
    return f'Processed payment for ${item.get_amount():,.2f}'

print(process_payment(Invoice(1500.0)))

Processed payment for $1,500.00


### 🔹 Data Modeling: `@dataclass`
Generates `__init__`, `__repr__`, `__eq__` boilerplate automatically based on type annotations. Object-Oriented Programming groups related data and functions together, making code modular, maintainable, and easy to scale. **Tip:** Always remember to include `self` as the first parameter in instance methods so Python knows which object instance is executing.

**Syntax:** `@dataclass class TransactionRecord: ...`


In [8]:
@dataclass
class TxData:
    id: str
    amount: float
    card: str

tx_inst = TxData(transactions[0]['transaction_id'], float(transactions[0]['transaction_amount']), transactions[0]['card_type'])
print('Dataclass instance:', tx_inst)

Dataclass instance: TxData(id='TX110686', amount=1216.33, card='Visa')


### 🔹 Immutable Data Modeling: `@dataclass(frozen=True)`
Enforces immutability: instance attribute mutation raises `FrozenInstanceError`. Object-Oriented Programming groups related data and functions together, making code modular, maintainable, and easy to scale. **Tip:** Always remember to include `self` as the first parameter in instance methods so Python knows which object instance is executing.

**Syntax:** `@dataclass(frozen=True) class ImmutableTx: ...`


In [9]:
@dataclass(frozen=True)
class FrozenTx:
    id: str
    amount: float

frozen_inst = FrozenTx('TX1', 100.0)
print('Frozen dataclass:', frozen_inst)

Frozen dataclass: FrozenTx(id='TX1', amount=100.0)


### 🔹 Value Constraints: `Literal`
Restricts variable or parameter to exact literal values. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `mode: Literal['r', 'w', 'a']`


In [10]:
def set_card(card: Literal['Visa', 'MasterCard', 'Amex', 'Discover']):
    return f'Card set to {card}'

print(set_card('Visa'))

Card set to Visa


### 🔹 Constant Protection: `Final`
Declares constants that static type checkers ensure are never reassigned or overridden. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `API_VERSION: Final[str] = 'v1.0'`


In [11]:
CURRENCY: Final[str] = 'USD'
print('Final constant:', CURRENCY)

Final constant: USD


### 🔹 Dictionary Schema: `TypedDict`
Defines expected dictionary key names and value types for static validation. Checking and validating data types prevents subtle runtime errors and ensures subsequent mathematical or string operations behave properly. **Tip:** Use `dict.get(key, default)` when looking up keys that might not exist, avoiding unexpected `KeyError` crashes.

**Syntax:** `class TxDict(TypedDict): id: str; amt: float`


In [12]:
class TransactionSchema(TypedDict):
    transaction_id: str
    transaction_amount: float

schema_obj: TransactionSchema = {
    'transaction_id': transactions[0]['transaction_id'],
    'transaction_amount': float(transactions[0]['transaction_amount'])
}
print('TypedDict schema object:', schema_obj)

TypedDict schema object: {'transaction_id': 'TX110686', 'transaction_amount': 1216.33}


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Nominal vs Structural Subtyping in Python

**Approach:** Explain difference between Nominal typing (`isinstance` / ABCs) and Structural typing (`typing.Protocol` / duck typing).
**Syntax:** `isinstance` vs `Protocol`


In [13]:
print('Nominal Subtyping: Explicit inheritance hierarchy (isinstance).')
print('Structural Subtyping: Shape and method signature matching (Protocol).')

Nominal Subtyping: Explicit inheritance hierarchy (isinstance).
Structural Subtyping: Shape and method signature matching (Protocol).
